In [ ]:
!pip install transformers datasets peft jiwer evaluate pandas numpy wandb

In [1]:
import math
import os
from collections import deque
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from accelerate import Accelerator
from accelerate.utils import set_seed
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, get_peft_model
from score import score_wer
from augment import Augmentor
import wandb
from torch.utils.data import DataLoader, SequentialSampler
from tqdm.auto import tqdm
from transformers import AutoModelForCTC, AutoProcessor, get_linear_schedule_with_warmup


KeyboardInterrupt: 

In [ ]:
# Core config (in-notebook)
BASE_MODEL_ID = "nvidia/parakeet-ctc-1.1b"
PROCESSOR_FROM_PRETRAINED_KWARGS = {}
DATASET_ID = "quinnlue/asr"
AUDIO_COLUMN = "audio_path"
TEXT_COLUMN = "orthographic_text"
OUTPUT_DIR = Path("artifacts/parakeet-ctc-lora")
SAMPLE_RATE = 16000

# Training hyperparameters
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.005
WARMUP_STEPS = 100
NUM_TRAIN_EPOCHS = 5
MAX_GRAD_NORM = 1.0
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.98
ADAM_EPSILON = 1e-8
GRADIENT_CHECKPOINTING = True
LOGGING_STEPS = 10
SAVE_STEPS = 1000
SEED = 42

# LoRA params
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
LORA_BIAS = "none"
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Runtime
DTYPE = torch.bfloat16

# Weights & Biases
WANDB_CFG = {
    "project": "parakeet-ctc-lora",
    "log_model": "false",
    "api_key_env_var": "WANDB_API_KEY",
}
WANDB_API_KEY_ENV_VAR = WANDB_CFG["api_key_env_var"]
if WANDB_API_KEY_ENV_VAR and os.getenv(WANDB_API_KEY_ENV_VAR):
    os.environ["WANDB_API_KEY"] = os.getenv(WANDB_API_KEY_ENV_VAR)

# Augmentation config (kept in notebook for future use)
AUGMENTATION_CFG = {
    "waveform": {
        "time_stretch_prob": 0.3,
        "rir_prob": 0.5,
        "noise_prob": 0.5,
        "min_speed": 0.9,
        "max_speed": 1.1,
        "max_stretch_samples": 480000,
        "min_snr_db": 5.0,
        "max_snr_db": 30.0,
    },
    "vtlp": {
        "prob": 0.3,
        "alpha_min": 0.9,
        "alpha_max": 1.1,
        "fhi": 4800.0,
    },
    "spec": {
        "prob": 1.0,
        "policy": "LB",
    },
    "noise_dataset": None,
    "rir_dataset": None,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if WANDB_CFG.get("project") is not None:
    os.environ["WANDB_PROJECT"] = str(WANDB_CFG["project"])
if WANDB_CFG.get("log_model") is not None:
    os.environ["WANDB_LOG_MODEL"] = str(WANDB_CFG["log_model"])

In [ ]:
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, **PROCESSOR_FROM_PRETRAINED_KWARGS)
model = AutoModelForCTC.from_pretrained(
    BASE_MODEL_ID,
    dtype=DTYPE,
    low_cpu_mem_usage=True,
)

ds = load_dataset(DATASET_ID)
train_dataset = ds["train"].filter(lambda x: x['audio_duration_sec'] < 15)
eval_dataset = ds["validation"].sort("audio_duration_sec")
test_dataset = ds["test"].sort("audio_duration_sec")

for name, split in [("train", train_dataset), ("eval", eval_dataset), ("test", test_dataset)]:
    print(name, len(split))

In [ ]:
@dataclass
class CTCDataCollator:
    processor: AutoProcessor
    audio_column: str
    text_column: str
    sampling_rate: int
    dtype: torch.dtype
    pad_token_id: int = 1024
    subsampling_factor: int = 8
    augmentor: Augmentor | None = None
    use_augment: bool = True

    def __call__(self, batch, use_augment: bool = True):
        use_augment = False

        # Decode audio and filter out corrupted samples in a single pass
        audio = []
        valid_batch = []
        for item in batch:
            try:
                arr = item[self.audio_column]["array"]
                audio.append(arr)
                valid_batch.append(item)
            except Exception as e:
                print(f"[CTCDataCollator] Skipping corrupted audio sample: {e}")
        if len(valid_batch) == 0:
            raise ValueError("Every sample in this batch has corrupted audio.")
        batch = valid_batch

        do_augment = use_augment and self.use_augment and (self.augmentor is not None)
        if do_augment:
            audio = [self.augmentor.augment_waveform(np.asarray(a, dtype=np.float32)) for a in audio]
        text = [item[self.text_column] for item in batch]

        inputs = self.processor(
            audio=audio,
            sampling_rate=self.sampling_rate,
            return_tensors="pt",
            padding=True,
        )

        if do_augment:
            inputs["input_features"] = self.augmentor.augment_features(
                inputs["input_features"],
                sampling_rate=self.sampling_rate,
            )

        labels = self.processor.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
        )

        label_ids = labels["input_ids"]
        label_ids = label_ids.masked_fill(labels["attention_mask"].ne(1), self.pad_token_id)

        n_frames = inputs["attention_mask"].sum(dim=-1)  # (B,)
        logit_lengths = n_frames // self.subsampling_factor
        target_lengths = (label_ids != self.pad_token_id).sum(dim=-1)  # (B,)

        keep = logit_lengths >= target_lengths
        if not keep.all():
            keep_idx = keep.nonzero(as_tuple=True)[0]
            if len(keep_idx) == 0:
                raise ValueError(
                    "Every sample in this batch violates the CTC length "
                    "constraint (label longer than downsampled input). "
                    "Consider filtering short audio from the dataset."
                )
            inputs["input_features"] = inputs["input_features"][keep_idx]
            inputs["attention_mask"] = inputs["attention_mask"][keep_idx]
            label_ids = label_ids[keep_idx]
        # ----------------------------------------------------------------------

        # Return CPU tensors; Accelerate handles device placement
        return {
            "input_features": inputs["input_features"].to(dtype=self.dtype),
            "attention_mask": inputs["attention_mask"],
            "labels": label_ids,
        }

In [ ]:
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias=LORA_BIAS,
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

In [ ]:
set_seed(SEED)

data_collator = CTCDataCollator(
    processor=processor,
    audio_column=AUDIO_COLUMN,
    text_column=TEXT_COLUMN,
    sampling_rate=SAMPLE_RATE,
    dtype=DTYPE,
    augmentor=Augmentor(AUGMENTATION_CFG),
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0,
)

eval_dataloader = DataLoader(
    eval_dataset,
    batch_size=EVAL_BATCH_SIZE,
    sampler=SequentialSampler(eval_dataset),
    collate_fn=data_collator,
    pin_memory=True,
    num_workers=0,
)

accelerator = Accelerator(
    mixed_precision="bf16",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(ADAM_BETA1, ADAM_BETA2),
    eps=ADAM_EPSILON,
)

num_update_steps_per_epoch = math.ceil(len(train_dataloader) / GRADIENT_ACCUMULATION_STEPS)
max_train_steps = NUM_TRAIN_EPOCHS * num_update_steps_per_epoch

lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=max_train_steps,
)

model, optimizer, train_dataloader, eval_dataloader, lr_scheduler = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader, lr_scheduler
)

@torch.no_grad()
def evaluate(model, eval_dataloader):
    """Run evaluation and return corpus-level WER."""
    model.eval()
    all_pred_texts = []
    all_label_texts = []

    for batch in tqdm(eval_dataloader, desc="Evaluating", leave=False):
        outputs = model(
            input_features=batch["input_features"],
            attention_mask=batch["attention_mask"],
        )
        pred_ids = torch.argmax(outputs.logits, dim=-1)

        pred_ids_np = accelerator.gather_for_metrics(pred_ids).cpu().numpy()
        all_pred_texts.extend(processor.batch_decode(pred_ids_np))  

        label_ids_np = accelerator.gather_for_metrics(batch["labels"]).cpu().numpy()
        label_ids_np[label_ids_np == -100] = processor.tokenizer.pad_token_id
        all_label_texts.extend(processor.tokenizer.batch_decode(label_ids_np, group_tokens=False))

    corpus_wer = float(score_wer(actual=all_label_texts, predicted=all_pred_texts))
    model.train()
    return corpus_wer

def save_checkpoint(model, global_step, save_dir, tag=None):
    """Save LoRA adapter weights."""
    name = tag if tag else f"checkpoint-step-{global_step}"
    ckpt_dir = save_dir / name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    unwrapped = accelerator.unwrap_model(model)
    unwrapped.save_pretrained(str(ckpt_dir))
    processor.save_pretrained(str(ckpt_dir))
    accelerator.print(f"Saved checkpoint to {ckpt_dir}")

print(f"Training for {NUM_TRAIN_EPOCHS} epochs, {max_train_steps} optimizer steps")
print(f"  Batches/epoch: {len(train_dataloader)}, grad accum: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Warmup steps: {WARMUP_STEPS}")

In [ ]:
wandb_run = None
if accelerator.is_main_process and WANDB_CFG.get("project"):
    if wandb is None:
        accelerator.print("wandb is not installed; skipping Weights & Biases logging.")
    else:
        wandb_run = wandb.init(
            project=WANDB_CFG["project"],
            config={
                # Model
                "base_model": BASE_MODEL_ID,
                "dataset": DATASET_ID,
                # Training
                "train_batch_size": TRAIN_BATCH_SIZE,
                "eval_batch_size": EVAL_BATCH_SIZE,
                "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
                "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "num_train_epochs": NUM_TRAIN_EPOCHS,
                "warmup_steps": WARMUP_STEPS,
                "max_train_steps": max_train_steps,
                "max_grad_norm": MAX_GRAD_NORM,
                "adam_beta1": ADAM_BETA1,
                "adam_beta2": ADAM_BETA2,
                "adam_epsilon": ADAM_EPSILON,
                "gradient_checkpointing": GRADIENT_CHECKPOINTING,
                "seed": SEED,
                "dtype": str(DTYPE),
                # LoRA
                "lora_r": LORA_R,
                "lora_alpha": LORA_ALPHA,
                "lora_dropout": LORA_DROPOUT,
                "lora_bias": LORA_BIAS,
                "lora_target_modules": LORA_TARGET_MODULES,
                # Data
                "train_samples": len(train_dataset),
                "eval_samples": len(eval_dataset),
                "sample_rate": SAMPLE_RATE,
                # Augmentation
                "augmentation": AUGMENTATION_CFG,
            },
        )
        # Log gradient & parameter histograms every 50 steps
        wandb.watch(accelerator.unwrap_model(model), log="all", log_freq=50)

global_step = 0
best_wer = float("inf")

for epoch in range(1, NUM_TRAIN_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    recent_wers = deque(maxlen=LOGGING_STEPS)  # rolling window of per-batch WERs
    progress = tqdm(train_dataloader, desc=f"Epoch {epoch}/{NUM_TRAIN_EPOCHS}", leave=True)

    for step, batch in enumerate(progress, start=1):
        # Original notebook ran forward pass in eval mode (BN / dropout frozen)
        model.eval()

        with accelerator.accumulate(model):
            outputs = model(
                input_features=batch["input_features"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            accelerator.backward(loss)

            if accelerator.sync_gradients:
                grad_norm = accelerator.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        # --- Greedy-decode this batch to get training WER ---
        with torch.no_grad():
            pred_ids = torch.argmax(outputs.logits.detach(), dim=-1)
            pred_ids_np = accelerator.gather_for_metrics(pred_ids).cpu().numpy()
            pred_texts = processor.batch_decode(pred_ids_np)

            label_ids_np = accelerator.gather_for_metrics(batch["labels"]).cpu().numpy()
            pad_id = processor.tokenizer.pad_token_id
            label_ids_np[label_ids_np == -100] = pad_id
            # Also mask out model pad_token_id (1024) so it doesn't decode as a real token
            label_ids_np[label_ids_np == 1024] = pad_id
            label_texts = processor.tokenizer.batch_decode(label_ids_np, group_tokens=False)

            batch_wer = float(score_wer(actual=label_texts, predicted=pred_texts))
            recent_wers.append(batch_wer)

        step_loss = loss.detach().float().item()
        epoch_loss += step_loss
        num_batches += 1

        if accelerator.sync_gradients:
            global_step += 1

            # Evaluate and save checkpoint every SAVE_STEPS
            if global_step % SAVE_STEPS == 0:
                eval_wer = evaluate(model, eval_dataloader)
                accelerator.print(f"  Step {global_step} | eval_wer={eval_wer:.4f}")
                if wandb_run is not None and accelerator.is_main_process:
                    wandb_run.log({"eval/wer": eval_wer, "global_step": global_step}, step=global_step)
                if eval_wer < best_wer:
                    best_wer = eval_wer
                    if wandb_run is not None and accelerator.is_main_process:
                        wandb_run.summary["best_eval_wer"] = float(best_wer)
                        wandb_run.summary["best_eval_step"] = int(global_step)
                if accelerator.is_main_process:
                    save_checkpoint(model, global_step, OUTPUT_DIR)

        if global_step % LOGGING_STEPS == 0:
            avg_loss = epoch_loss / num_batches
            avg_train_wer = sum(recent_wers) / len(recent_wers)
            lr = lr_scheduler.get_last_lr()[0]
            progress.set_postfix(loss=f"{avg_loss:.4f}", wer=f"{avg_train_wer:.4f}", lr=f"{lr:.2e}", step=global_step)
            if wandb_run is not None and accelerator.is_main_process:
                wandb_run.log({
                    "train/loss": float(avg_loss),
                    "train/step_loss": float(step_loss),
                    "train/wer": float(avg_train_wer),
                    "train/lr": float(lr),
                    "train/grad_norm": float(grad_norm) if isinstance(grad_norm, (int, float)) else float(grad_norm.item()),
                    "train/epoch": int(epoch),
                    "global_step": int(global_step),
                }, step=global_step)

    # End of epoch: evaluate
    eval_wer = evaluate(model, eval_dataloader)
    avg_epoch_loss = epoch_loss / max(num_batches, 1)

    if wandb_run is not None and accelerator.is_main_process:
        wandb_run.log({
            "train/epoch_loss": float(avg_epoch_loss),
            "eval/wer": eval_wer,
            "global_step": global_step,
        }, step=global_step)

    accelerator.print(
        f"Epoch {epoch} | train_loss={avg_epoch_loss:.4f} | eval_wer={eval_wer:.4f}"
    )

    if eval_wer < best_wer:
        best_wer = eval_wer
        if wandb_run is not None and accelerator.is_main_process:
            wandb_run.summary["best_eval_wer"] = float(best_wer)
            wandb_run.summary["best_eval_epoch"] = int(epoch)
            wandb_run.summary["best_eval_step"] = int(global_step)

# Final save
if accelerator.is_main_process:
    save_checkpoint(model, global_step, OUTPUT_DIR, tag="checkpoint-final")

if wandb_run is not None and accelerator.is_main_process:
    wandb_run.summary["final_train_loss"] = float(avg_epoch_loss)
    wandb_run.summary["final_eval_wer"] = float(eval_wer)
    wandb_run.log({"best/eval_wer": float(best_wer), "global_step": int(global_step)}, step=global_step)
    wandb.unwatch(accelerator.unwrap_model(model))
    wandb_run.finish()

accelerator.print(f"\nTraining complete. Best WER: {best_wer:.4f}")
